<a href="https://colab.research.google.com/github/astronerd21/Deep-Learning-Oil-Slick-Detection/blob/main/Lookalike_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# False Positive / Lookalike Analysis
**Goal:** Inspect "NO OIL" (0) samples that the model incorrectly predicted as "OIL" (1). This notebook performs an automated evaluation to identify false positives and visualizes their radar signatures.

**Methodology:** Analysis of false positive IDs exported from the model evaluation pipeline, followed by a comparative visualization of VV and VH SAR channel signatures and their pixel intensity distributions (histograms).

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

%pip install rasterio matplotlib numpy

os.makedirs('/content/data', exist_ok=True)

if not os.path.exists('/content/data/images_s1'):
    print("Extracting satellite images from Drive...")
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-00.tar -C /content/data/
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-01.tar -C /content/data/
    print("Extraction complete!")
else:
    print("Images are already extracted.")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths to the finished evaluations
fp_paths = {
    "Baseline Random": Path("/content/drive/MyDrive/OilSlickProject/data/OilSlick/results/baseline_random/evaluation_outputs/false_positives_test_clean.txt"),
    "ResNet Geo OOD": Path("/content/drive/MyDrive/OilSlickProject/data/OilSlick/results/geographic_ood/evaluation_outputs/false_positives_geo_test_clean.txt"),
    "TerraMind Geo OOD": Path("/content/drive/MyDrive/OilSlickProject/data/OilSlick/results/terramind_ood/evaluation_outputs/false_positives_geo_test_clean.txt")
}

# Function to read the lists
def load_fp_list(path):
    if not path.exists():
        print(f"WARNING: File not found -> {path}")
        return []
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

# Load lists into a dictionary
fps = {name: load_fp_list(path) for name, path in fp_paths.items()}

for name, ids in fps.items():
    print(f"{name}: {len(ids)} False Positives loaded.")

# Helper functions for plotting
data_dir = Path("/content/data/images_s1/")

def load_and_normalize_sar(filepath: Path) -> np.ndarray:
    with rasterio.open(filepath) as src:
        data = src.read().astype(np.float32)
    for c in range(data.shape[0]):
        channel_mean = np.mean(data[c])
        channel_std = np.std(data[c])
        if channel_std > 0:
            data[c] = (data[c] - channel_mean) / channel_std
    return data

def plot_top_5(model_name):
    ids = fps.get(model_name, [])[:5]  # Hier wird auf 5 abgeschnitten
    if not ids:
        return

    num_samples = len(ids)
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 4 * num_samples))
    fig.suptitle(f"Lookalike Analysis: {model_name} (Top {num_samples})", fontsize=16, y=1.02)

    if num_samples == 1:
        axes = np.array([axes])

    for idx, filename in enumerate(ids):
        # Fallback in case the .txt files are missing the .tif extension
        img_file = filename if filename.endswith('.tif') else f"{filename}_s1.tif"
        filepath = data_dir / img_file

        try:
            data = load_and_normalize_sar(filepath)
            axes[idx, 0].imshow(data[0], cmap='gray'); axes[idx, 0].set_title(f"VV | {filename}"); axes[idx, 0].axis('off')
            axes[idx, 1].hist(data[0].ravel(), bins=50, color='blue', alpha=0.7); axes[idx, 1].set_title("VV Hist")
            axes[idx, 2].imshow(data[1], cmap='gray'); axes[idx, 2].set_title(f"VH | {filename}"); axes[idx, 2].axis('off')
            axes[idx, 3].hist(data[1].ravel(), bins=50, color='orange', alpha=0.7); axes[idx, 3].set_title("VH Hist")
        except Exception as e:
            axes[idx, 0].set_title(f"Error at {filename}"); axes[idx, 0].axis('off')
            print(f"Error loading {filename}: {e}")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_top_5("Baseline Random")

**Physical Classification (Baseline Random):**

* **Dominant Phenomenon:** The baseline model relies heavily on simple pixel intensity and lacks spatial understanding, making it susceptible to a wide range of standard lookalikes.
* **Specific Observations:** * **Uniform Calm Water:** It fails on plain, uniform sea surfaces with low backscatter (e.g., `neg_00033`, `neg_00212`).
    * **Biogenic Films:** Images like `neg_00094` and `neg_00248` show the classic swirling eddies and streaks of natural organic films that the model mistakes for oil.
    * **Complex Artifacts:** Image `neg_00214` displays a very dark background with bright spots (likely ships or artifacts), causing a false positive due to the overall low VV signal.
* **Model Behavior:** The baseline has not learned structural features. It classifies anything with a significantly low VV-channel backscatter as oil, whether it's a biogenic swirl or just a windless patch of ocean.

In [ ]:
plot_top_5("ResNet Geo OOD")

**Physical Classification (ResNet Geo OOD):**

* **Dominant Phenomenon:** Structural lookalikes and boundaries. The model recognizes shapes and edges but lacks the broader geographic context to correctly classify complex natural formations.
* **Specific Observations:**
    * **Linear Structures & Wakes:** It gets tricked by highly structured ship wakes (`neg_00342`) and linear biogenic streaks (`neg_00113`, `neg_00609`).
    * **Land/Sea Boundaries:** Image `neg_00311` shows the model struggling with a coastal wind shadow or the edge of a landmass.
    * **Artifacts:** Image `neg_00109` features a dark background with bright reflection artifacts, which the convolutions fail to filter out.
* **Model Behavior:** The ResNet is a step up from the baseline as it utilizes spatial convolutions to find "slick-like" shapes. However, its limited receptive field means it cannot reliably differentiate between the structural edge of a natural phenomenon and an actual oil spill.

In [ ]:
plot_top_5("TerraMind Geo OOD")

**Physical Classification (TerraMind Geo OOD):**

* **Dominant Phenomenon:** Morphologically complex, "true" physical lookalikes. The Foundation Model mainly fails when natural phenomena perfectly mimic the physical and structural signature of an oil spill.
* **Specific Observations:**
    * **Extreme Complexity:** Image `neg_00092` shows an incredibly complex swirling pattern (massive biogenic film or eddies) that physically dampens the radar signal exactly like oil. Image `neg_00136` displays a highly unusual, mottled texture.
    * **Overlap with ResNet:** The model still shares some false positives with the ResNet, specifically the dark artifact image (`neg_00109`), biogenic streaks (`neg_00113`), and coastal boundaries (`neg_00254`).
* **Model Behavior:** While the Foundation Model demonstrates a robust understanding of standard SAR physics, it proves that single-modality SAR data has a hard physical limit. When lookalikes (like dense algal blooms) dampen capillary waves in the exact same morphological patterns as mineral oil, external context (like optical data) is required for accurate classification.

In [ ]:
# Convert lists to sets to find the intersection
sets = {name: set(ids) for name, ids in fps.items()}

# Intersection: Images that were a False Positive in ALL THREE models
uncommon_lookalikes = sets["Baseline Random"] & sets["ResNet Geo OOD"] & sets["TerraMind Geo OOD"]

print("=== OVERLAP ANALYSIS ===")
print(f"Number of 'impossible lookalikes' (False alarm in all 3 models): {len(uncommon_lookalikes)}\n")

print("Top 10 IDs of these hard cases:")
for img_id in list(uncommon_lookalikes)[:10]:
    print(f" - {img_id}")

# visualize the hardest case
if uncommon_lookalikes:
    hard_case = list(uncommon_lookalikes)[0]
    print(f"\nVisualizing an exemplary hard case: {hard_case}")
    fps["Hardcase Demo"] = [hard_case]
    plot_top_5("Hardcase Demo")

In [ ]:
%cd /content
!rm -rf Deep-Learning-Oil-Slick-Detection
!git clone https://github.com/astronerd21/Deep-Learning-Oil-Slick-Detection.git
%cd Deep-Learning-Oil-Slick-Detection

print("\n Contents of Folder")
!ls src/

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import ResNet18_Weights, ResNet50_Weights, resnet18, resnet50
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import rasterio

class SARResNet(nn.Module):
    _BACKBONES = {
        "resnet18": (resnet18, ResNet18_Weights.DEFAULT),
        "resnet50": (resnet50, ResNet50_Weights.DEFAULT),
    }

    def __init__(self, backbone: str = "resnet18", pretrained: bool = True, in_channels: int = 2) -> None:
        super().__init__()
        factory, weights = self._BACKBONES[backbone]
        net = factory(weights=weights if pretrained else None)

        orig_conv = net.conv1
        new_conv = nn.Conv2d(
            in_channels, orig_conv.out_channels, kernel_size=orig_conv.kernel_size,
            stride=orig_conv.stride, padding=orig_conv.padding, bias=orig_conv.bias is not None,
        )

        if pretrained:
            with torch.no_grad():
                mean_weight = orig_conv.weight.mean(dim=1, keepdim=True)
                new_conv.weight.copy_(mean_weight.expand(-1, in_channels, -1, -1))
                if orig_conv.bias is not None and new_conv.bias is not None:
                    new_conv.bias.copy_(orig_conv.bias)
        net.conv1 = new_conv

        in_features = net.fc.in_features
        net.fc = nn.Linear(in_features, 1)
        self.net = net

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)

class GradCAM:
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_heatmap(self, input_tensor: torch.Tensor) -> np.ndarray:
        self.model.eval()
        self.model.zero_grad()
        output = self.model(input_tensor)
        output.backward(retain_graph=True)

        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.squeeze(0)

        for i in range(activations.shape[0]):
            activations[i, :, :] *= pooled_gradients[i]

        heatmap = torch.mean(activations, dim=0).squeeze()
        heatmap = F.relu(heatmap)

        if torch.max(heatmap) > 0:
            heatmap /= torch.max(heatmap)

        heatmap = heatmap.unsqueeze(0).unsqueeze(0)
        heatmap = F.interpolate(heatmap, size=(input_tensor.size(2), input_tensor.size(3)), mode='bilinear', align_corners=False)

        return heatmap.squeeze().detach().cpu().numpy()

def generate_heatmap(input_tensor: torch.Tensor, model: torch.nn.Module) -> np.ndarray:
    target_layer = model.net.layer4[-1]
    grad_cam = GradCAM(model, target_layer)
    return grad_cam.generate_heatmap(input_tensor)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SARResNet(backbone="resnet18", in_channels=2).to(device)
model_path = "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/resnet_baseline.pt"

try:
    checkpoint = torch.load(model_path, map_location=device)

    # Extract the correct dictionary key if it exists
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")

gradcam_ids = ["neg_00109", "neg_00214", "neg_00311"]
data_dir = Path("/content/data/images_s1/")

fig, axes = plt.subplots(len(gradcam_ids), 3, figsize=(15, 5 * len(gradcam_ids)))
fig.suptitle("Grad-CAM Explainability (ResNet Geo OOD)", fontsize=16, y=1.02)

for idx, img_id in enumerate(gradcam_ids):
    filename = f"{img_id}_s1.tif" if not img_id.endswith(".tif") else img_id
    filepath = data_dir / filename

    try:
        with rasterio.open(filepath) as src_img:
            data = src_img.read().astype(np.float32)

        for c in range(data.shape[0]):
            c_mean, c_std = np.mean(data[c]), np.std(data[c])
            if c_std > 0:
                data[c] = (data[c] - c_mean) / c_std

        tensor_data = torch.tensor(data).unsqueeze(0).to(device)
        heatmap = generate_heatmap(tensor_data, model)

        axes[idx, 0].imshow(data[0], cmap='gray')
        axes[idx, 0].set_title(f"VV | {img_id}")
        axes[idx, 0].axis('off')

        axes[idx, 1].imshow(data[1], cmap='gray')
        axes[idx, 1].set_title(f"VH | {img_id}")
        axes[idx, 1].axis('off')

        axes[idx, 2].imshow(data[0], cmap='gray')
        axes[idx, 2].imshow(heatmap, cmap='jet', alpha=0.5)
        axes[idx, 2].set_title("Grad-CAM Focus")
        axes[idx, 2].axis('off')

    except Exception as e:
        print(f"Error processing {img_id}: {e}")

plt.tight_layout()
plt.show()